In [0]:
from pyspark.sql.functions import *;


day4=spark.read.format("json").option("multiline","true").option("inferSchema","true").load("/Volumes/workspace/default/json/ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 100500409-07-2025.json")

day4.printSchema()
daily_sales = (
    day4.groupBy("businessDate","deployment_name","deployment_id")
      .agg(sum(col("aggregation.netAmount")).alias("net_sales"))
)

daily_sales.display()


In [0]:
# day4.select("payments").display()
# day4.select("_cash").display()

exploded_cards=day4.withColumn("cards",explode("payments.cards"))

exploded_cash = day4.withColumn(
    "cash_amount",
    explode(col("payments.cash"))
)

total_amount_cards = exploded_cards.groupBy(
    col("cards.cardType").alias("payment_type")
).agg(
    sum(col("cards.totalAmount").cast("double")).alias("total_amount")
)

total_card_payment=total_amount_cards.filter("total_amount > 0")
total_card_payment.display()

total_cash_amount = exploded_cash.agg(
    sum(col("cash_amount")).alias("total_amount")
)

total_cash_payment=total_cash_amount.withColumn("payment_type",lit("cash"))

display(total_cash_payment)

In [0]:
# find total payment amount includes card payment and cash payment
total_payment = total_card_payment.unionByName(total_cash_payment,allowMissingColumns=True)
total_payment.display()

In [0]:
total_payment=total_payment.withColumn("deployment_id",lit("64df2c3cfadc7fca6133d2b1"))\
    .withColumn("deployment_name",lit("ALBAIK - SQ DB04 - AJMAN CITY CENTRE - 1005004"))
total_payment.display()

In [0]:
#item wise sales
kots=day4.select(explode("_kots"))
kots.display()

In [0]:
items=day4.select(explode("_kots.items").alias("items"))
items.display()
items.printSchema()

In [0]:
items.printSchema()


items=items.select(
    col("items._id").alias("item_id"),
    col("items.name").alias("item_name"),
    col("items.quantity").alias("qty"),
    col("items.rate").alias("rate"),
    col("items.subtotal").alias("subtotal")
)
items.display()

In [0]:
items_exploded=items.withColumn("qty_elements",explode(col("qty")))\
    .withColumn("total",explode(col("subtotal")))

order_sales=items_exploded.groupBy("item_id","item_name").agg(sum(col("qty_elements")).cast("bigint").alias("total_quantity"), sum(col("total")).cast("double").alias("total sales"))
order_sales.display()

In [0]:
items_exploded = items.withColumn(
    "item",
    explode(arrays_zip("item_id", "item_name", "qty", "rate", "subtotal"))
)

items_exploded.display()

In [0]:
item_sales = items_exploded.select(
    "item.item_id",
    "item.item_name",
    "item.qty",
    "item.rate",
    "item.subtotal"
)

item_sales.display()

In [0]:
item_wise_sales = item_sales.groupBy("item_id", "item_name") \
                            .agg(sum("subtotal").alias("total_sales"))
item_wise_sales.display()

In [0]:
# Drop the duplicate deployment_name column from bill_net before the joifrom pyspark.sql import functions as F
from pyspark.sql import functions as F
aggre_exploded=day4.select("aggregation.netRoundedAmount","aggregation.netAmount")
aggre_exploded.display()

bill_net = day4.withColumn(
    "NetSalesBill",
    F.when(
        F.col("aggregation.netRoundedAmount").isNotNull(),
        F.col("aggregation.netRoundedAmount")
    ).otherwise(F.col("aggregation.netAmount"))
).select(
    "billNumber",
    "NetSalesBill",
    F.col("businessDate").alias("bill_businessDate"),  # Rename here
    "deployment_id"
)

bill_net.display()


In [0]:
flat = day4 \
    .withColumn("kot", explode("_kots")) \
    .withColumn("item", explode("kot.items")) \
    .withColumn("item_tax", explode_outer("item.taxes")) \
    .withColumn("brand_id", col("enterpriseDetails.brand_id")) \
    .withColumn("brand_name", lit("Albaik UAE")) \
    .withColumn("StoreCode", regexp_extract("deployment_name", r"(\d+)$", 1)) \
    .withColumn("country_name", lit("United Arab Emirates"))


#bill_net = bill_net.drop("deployment_name")

flat.display()

In [0]:
bill_summary = flat.groupBy(
    "billNumber",
    "businessDate",
    "StoreCode",
    "deployment_name",
    "brand_name",
    "country_name"
).agg(
    F.countDistinct("kot.kotNumber").alias("TotalKOTs"),
    F.sum("item.quantity").alias("TotalItems"),
    F.sum("item.subtotal").alias("TotalAmount"),
    F.sum("item_tax.tax_amount").alias("TotalTax"),
    F.sum("kot.totalDiscount").alias("TotalDiscount"),
    F.sum("aggregation.roundOff").alias("RoundOff")
).join(bill_net, on="billNumber", how="left")

bill_summary.display()

In [0]:
mysummary_correct =flat.groupBy(
    "businessDate",
    "StoreCode",
    "deployment_name",
    "brand_name",
    "country_name"
).agg(
    F.countDistinct("billNumber").alias("TotalBills"),
    F.sum("TotalItems").alias("TotalItems"),
    F.sum("TotalAmount").alias("TotalAmount"),
    F.sum("TotalTax").alias("TotalTax"),
    F.sum("TotalDiscount").alias("TotalDiscount"),
    F.sum("RoundOff").alias("RoundOff"),
    F.sum("NetSalesBill").alias("NetSales")
)

mysummary_correct.display()

In [0]:
from pyspark.sql.functions import *

path="/Volumes/workspace/default/projectfiles"
df=spark.read.format("json").option("multiline","true").option("inferSchema","true").load(path)
df.display()

daily_sales = (
    df.groupBy("businessDate","deployment_name","deployment_id")
      .agg(sum(col("aggregation.netAmount")).alias("net_sales"))
)

df.select("businessDate").distinct().show()

daily_sales.display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
df.select("businessDate").distinct().show(50, False)
# ------------------------------------------
# 1. FLAG VOID BILLS
# ------------------------------------------
bill_flag = df.withColumn(
    "isVoidBill",
    F.expr("exists(_kots, x -> x.isVoid = true)")
).select("billNumber", "isVoidBill")

# ------------------------------------------
# 2. BILL-LEVEL NET SALES (NO EXPLODE HERE)
# ------------------------------------------
bill_net = df.withColumn(
    "NetSalesBill",
    F.when(F.col("aggregation.netRoundedAmount").isNotNull(),
           F.col("aggregation.netRoundedAmount"))
     .otherwise(F.col("aggregation.netAmount"))
).withColumn(
    "StoreCode", F.regexp_extract("deployment_name", r"(\d+)$", 1)
).withColumn(
    "brand_name", F.lit("Albaik UAE")
).withColumn(
    "country_name", F.lit("United Arab Emirates")
).select(
    "billNumber", "businessDate", "deployment_name", "StoreCode",
    "brand_name", "country_name", "NetSalesBill",
    F.col("aggregation.roundOff").alias("roundOff")
)

# ------------------------------------------
# 3. JOIN FLAG + REMOVE VOID BILLS
# ------------------------------------------
bill_clean = bill_net.join(bill_flag, "billNumber", "left") \
                     .filter(F.col("isVoidBill") == False)

# ------------------------------------------
# 4. BILL-LEVEL SUMMARY (Correct NetSales)
# ------------------------------------------
bill_summary = bill_clean.groupBy(
    "businessDate", "deployment_name", "StoreCode", "brand_name", "country_name"
).agg(
    F.countDistinct("billNumber").alias("TotalBills"),
    F.sum("NetSalesBill").alias("NetSales"),
    F.sum("roundOff").alias("RoundOff")
)

# ------------------------------------------
# 5. FLATTEN ONLY FOR ITEMS, TAX, DISCOUNT
# ------------------------------------------
flat = df.join(bill_flag, "billNumber", "left") \
         .filter(F.col("isVoidBill") == False) \
         .withColumn("StoreCode", F.regexp_extract("deployment_name", r"(\d+)$", 1)) \
         .withColumn("brand_name", F.lit("Albaik UAE")) \
         .withColumn("country_name", F.lit("United Arab Emirates")) \
         .withColumn("kot", F.explode("_kots")) \
         .withColumn("item", F.explode("kot.items")) \
         .withColumn("tax", F.explode_outer("item.taxes"))

# ------------------------------------------
# 6. ITEM-LEVEL SUMMARY
# ------------------------------------------
item_summary = flat.groupBy(
    "businessDate", "deployment_name", "StoreCode", "brand_name", "country_name"
).agg(
    F.sum("item.quantity").alias("TotalItems"),
    F.sum("item.subtotal").alias("TotalAmount"),
    F.sum("tax.tax_amount").alias("TotalTax"),
    F.sum("kot.totalDiscount").alias("TotalDiscount")
)

# ------------------------------------------
# 7. FINAL MERGE
# ------------------------------------------
final_summary = (
    bill_summary.join(
        item_summary,
        ["businessDate", "deployment_name", "StoreCode", "brand_name", "country_name"],
        "left"
    )
)

final_summary.display()



